# 🤟 Sign Language Translator – End-to-End Notebook

This notebook contains setup, preprocessing, model training, evaluation, and notes for real-time inference.


## 1️⃣ Setup

In [ ]:

# Install dependencies (run once)
# !pip install tensorflow opencv-python mediapipe streamlit numpy scikit-learn matplotlib pyttsx3

import os, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt


## 2️⃣ Paths & Config

In [ ]:

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10

TRAIN_DIR = "data/train"
VAL_DIR = "data/val"
TEST_DIR = "data/test"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)


## 3️⃣ Data Generators (Preprocessing)

In [ ]:

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)
val_gen = val_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

num_classes = train_gen.num_classes
class_indices = train_gen.class_indices
with open(os.path.join(MODEL_DIR, "class_labels.json"), "w") as f:
    json.dump(class_indices, f)
print("Classes:", class_indices)


## 4️⃣ Build & Train CNN Model

In [ ]:

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)
model.save(os.path.join(MODEL_DIR, "sign_model.h5"))


## 5️⃣ Plot Training Curves

In [ ]:

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend(); plt.title("Loss")
plt.show()


## 6️⃣ Evaluation

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix

model = load_model(os.path.join(MODEL_DIR, "sign_model.h5"))
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

# Load labels
with open(os.path.join(MODEL_DIR, "class_labels.json")) as f:
    class_indices = json.load(f)
inv_labels = {v:k for k,v in class_indices.items()}
labels = [inv_labels[i] for i in range(len(inv_labels))]

print(classification_report(y_true, y_pred, target_names=labels))
cm = confusion_matrix(y_true, y_pred)
cm


## 7️⃣ Real-time Inference (Note)
Webcam inference works best via a Python script (OpenCV window). See `scripts/predict_webcam.py` in the repo for real-time demo.

## 8️⃣ Tips
- Increase epochs for better accuracy
- Use MobileNet/EfficientNet for transfer learning
- Add MediaPipe hand cropping for robustness
